# Variable selection: lab notebook

**Statistical Modelling and Forecasting, Regression and exponential smoothing**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import itertools
import numpy as np
import pandas as pd
import statsmodels.api as sm

### 1. Longley data: employment explained by up to six candidate predictors

In [ ]:
# 1. Longley data: employment explained by up to six candidate predictors
L = sm.datasets.longley.load_pandas().data
y = L["TOTEMP"]
cand = ["GNPDEFL", "GNP", "UNEMP", "ARMED", "POP", "YEAR"]
n = len(y)
SST = ((y - y.mean()) ** 2).sum()
full = sm.OLS(y, sm.add_constant(L[cand])).fit()
s2full = full.mse_resid
print("n =", n, "| full model: R2 =", round(full.rsquared, 4), "| sigma^2 =", round(s2full / 1e6, 4), "(millions^2)")

### 2. Best-subset search: every subset of the six predictors

In [ ]:
# 2. Best-subset search: every subset of the six predictors
rows = []
for k in range(1, 7):
    for sub in itertools.combinations(cand, k):
        m = sm.OLS(y, sm.add_constant(L[list(sub)])).fit()
        p = k + 1
        rows.append(dict(vars=",".join(sub), k=k, p=p, SSE=m.ssr, R2=m.rsquared, adjR2=m.rsquared_adj,
                         Cp=m.ssr / s2full - (n - 2 * p), AIC=n * np.log(m.ssr / n) + 2 * p, BIC=n * np.log(m.ssr / n) + p * np.log(n)))
T = pd.DataFrame(rows)
print("subsets examined:", len(T))
best = T.loc[T.groupby("k")["SSE"].idxmin(), ["vars", "k", "R2", "adjR2", "Cp", "AIC", "BIC"]]
print(best.round(3).to_string(index=False))
for crit, rule in [("adjR2", "max"), ("Cp", "min"), ("AIC", "min"), ("BIC", "min")]:
    i = T[crit].idxmax() if rule == "max" else T[crit].idxmin()
    print(f"{crit:6s} chooses: {T.loc[i, 'vars']}")

### 3. Forward selection by AIC

In [ ]:
# 3. Forward selection by AIC
chosen, left = [], cand.copy()
cur = n * np.log(((y - y.mean()) ** 2).sum() / n) + 2
while left:
    trial = {v: (lambda m: n * np.log(m.ssr / n) + 2 * (len(chosen) + 2))(sm.OLS(y, sm.add_constant(L[chosen + [v]])).fit()) for v in left}
    v = min(trial, key=trial.get)
    if trial[v] >= cur:
        break
    chosen.append(v); left.remove(v); cur = trial[v]
    print("added", v, "-> AIC", round(cur, 2))
print("forward selection result:", chosen)

### 4. Mallows Cp for the full model is exactly p

In [ ]:
# 4. Mallows Cp for the full model is exactly p
print("Cp of the full model:", round(T.loc[T.k == 6, "Cp"].iloc[0], 3), "= p =", 7)

### 5. The numerical problem

In [ ]:
# 5. The numerical problem
nn, sst = 40, 1000
S = [450, 250, 230, 225]; P = [2, 3, 4, 5]
print("R2 of M2:", round(1 - S[1] / sst, 4), "| adjusted R2 of M3:", round(1 - (S[2] / (nn - 4)) / (sst / (nn - 1)), 4))
print("Cp of M2:", round(S[1] / (S[3] / (nn - 5)) - (nn - 6), 3), "| AIC of M3:", round(nn * np.log(S[2] / nn) + 8, 3))
print("BIC:", np.round([nn * np.log(s / nn) + p * np.log(nn) for s, p in zip(S, P)], 2))

## Expected output

Running all cells above should print:

```
n = 16 | full model: R2 = 0.9955 | sigma^2 = 0.0929 (millions^2)
subsets examined: 63
                            vars  k    R2  adjR2     Cp     AIC     BIC
                             GNP  1 0.967  0.965 52.949 209.451 210.996
                      UNEMP,YEAR  2 0.982  0.980 25.208 201.654 203.972
                UNEMP,ARMED,YEAR  3 0.993  0.991  6.239 189.170 192.260
            GNP,UNEMP,ARMED,YEAR  4 0.995  0.994  3.239 184.249 188.112
        GNP,UNEMP,ARMED,POP,YEAR  5 0.995  0.993  5.031 185.885 190.520
GNPDEFL,GNP,UNEMP,ARMED,POP,YEAR  6 0.995  0.992  7.000 187.829 193.237
adjR2  chooses: GNP,UNEMP,ARMED,YEAR
Cp     chooses: GNP,UNEMP,ARMED,YEAR
AIC    chooses: GNP,UNEMP,ARMED,YEAR
BIC    chooses: GNP,UNEMP,ARMED,YEAR
added GNP -> AIC 209.45
added UNEMP -> AIC 203.09
added ARMED -> AIC 200.91
added YEAR -> AIC 184.25
forward selection result: ['GNP', 'UNEMP', 'ARMED', 'YEAR']
Cp of the full model: 7.0 = p = 7
R2 of M2: 0.75 | adjusted R2 of M3: 0.7508
Cp of M2: 4.889 | AIC of M3: 77.968
BIC: [104.19  84.37  84.72  87.53]
```

## Practice exercises

Try these in the cells below.

1. With ρ = 0, look at which subsets AIC and BIC choose. Do they contain exactly the true predictors?
2. Raise the correlation between predictors to 0.8. Press New sample several times. Does the chosen set become unstable?
3. Lower n to 30. Which criterion tends to keep extra variables?

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
